# The Fermi-Pasta-Ulam Problem

In 1953, at Los Alamos, Enrico Fermi, John Pasta, Stanislaw Ulam, and Mary
Tsingou ran one of the very first physics experiments on a digital computer.
They modeled a chain of masses joined by springs and gave it a small
*nonlinear* stiffness. Fermi's expectation was simple and, at the time,
"obvious": if you dump all the energy into the slowest vibration mode of the
chain and let the nonlinearity mix the modes together, the energy should slowly
**thermalize** - spread out evenly across every mode, the way heat spreads
through a solid.

That is not what happened. Instead, after wandering into a handful of higher
modes, the energy came flooding **back** into the original mode almost
completely. The chain kept *remembering* its initial state and returning to it.
This surprising, near-periodic behavior is now called the **FPU recurrence**,
and it launched the modern fields of nonlinear dynamics and soliton theory.

This notebook builds up to that result in two stages:

1. **A linear spring-mass chain (5 masses).** With no nonlinearity the chain is
   a textbook coupled oscillator. We integrate it with a symplectic method and
   use an FFT to recover its **normal modes** - the pure vibration frequencies
   that a linear chain never mixes.
2. **The nonlinear FPU-$\alpha$ chain (32 masses).** We switch on the quadratic
   force correction, excite only the lowest mode, and watch the energy leave and
   then **recur** - reproducing the 1953 result.

Both stages use the same **Yoshida 4th-order symplectic integrator** from the
companion notebook `pendulums.ipynb`, because tracking energy flow between modes
over long times demands an integrator that does not leak energy of its own.

---
## Setup: the symplectic integrator

Everything the notebook needs is defined locally so it runs stand-alone. The one
piece we borrow in spirit from `pendulums.ipynb` is `yoshida_coeffs()`, which
returns the position substep weights $c$ and velocity substep weights $d$ for
Yoshida's 4th-order method. A single step is composed of three leapfrog-like
substeps:

$$u \mathrel{+}= c_0\,v\,\Delta t,\qquad
  \bigl[\;v \mathrel{+}= d_j\,a(u)\,\Delta t,\quad
         u \mathrel{+}= c_{j+1}\,v\,\Delta t\;\bigr]_{j=0,1,2}.$$

The middle substep uses **negative** weights ($c_2, c_3, d_2 < 0$), stepping
briefly backward in time to cancel the lower-order error left by a single
leapfrog step. That cancellation is what makes the method 4th-order accurate
while staying symplectic.

In [ ]:
"""fpu.ipynb"""

# Cell 01 - Imports and the Yoshida symplectic coefficients

%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
from numpy.fft import fft, fftfreq
from scipy.signal import find_peaks


def yoshida_coeffs() -> tuple[np.ndarray, np.ndarray]:
    """Position (c) and velocity (d) substep weights for Yoshida 4th order."""
    cbrt2 = 2.0 ** (1.0 / 3.0)
    w1 = 1.0 / (2.0 - cbrt2)
    w0 = -cbrt2 / (2.0 - cbrt2)
    c = np.array([w1 / 2.0, (w0 + w1) / 2.0, (w0 + w1) / 2.0, w1 / 2.0])
    d = np.array([w1, w0, w1])
    return c, d


# Quick check: the weights must sum to 1 (one full step of length dt)
c, d = yoshida_coeffs()
print(f"c = {c}")
print(f"d = {d}")
print(f"sum(c) = {c.sum():.6f}  (expected 1)")
print(f"sum(d) = {d.sum():.6f}  (expected 1)")

---
# Part 1 - A Linear Spring-Mass Chain

We start with the simplest version of the FPU setup: $N = 5$ equal masses in a
line, each joined to its two nearest neighbors by an ideal spring, with the two
ends pinned to **fixed walls**. Let $u_i$ be the displacement of mass $i$ from
its equilibrium position. The walls contribute the boundary conditions
$u_0 = u_{N+1} = 0$.

Each spring obeys **Hooke's law**: the force it exerts is proportional to how far
it is stretched. Mass $i$ feels its right spring pulling it one way and its left
spring pulling the other, so its acceleration is

$$m\,\ddot{u}_i = k\,(u_{i+1} - u_i) - k\,(u_i - u_{i-1})
             = k\,(u_{i+1} - 2u_i + u_{i-1}).$$

This is **nearest-neighbor coupling**: a mass only ever talks to its immediate
neighbors. (The `acceleration` function below also carries an $\alpha$ term for
the nonlinear correction; here we set $\alpha = 0$ so it reduces to pure Hooke's
law, and we will switch it on in Part 2.)

We pick a stiff chain ($k = 50\ \text{N/m}$, $m = 1\ \text{kg}$), pluck only the
first mass forward, and let the symplectic integrator carry the motion.

In [ ]:
# Cell 02 - Linear chain parameters, acceleration, and the integrator

# Physical parameters for the linear chain
N = 5  # number of masses
K = 50.0  # spring constant (N/m)
M = 1.0  # mass of each particle (kg)
ALPHA = 0.0  # FPU nonlinearity strength (0 = pure Hooke's law)
A = 1.0  # equilibrium spacing (m); walls at x=0 and x=(N+1)*A

# Padded position array shared by acceleration(): index 0 and -1 are the walls
_u_ext = np.empty(N + 2)
_u_ext[0] = 0.0  # left wall (fixed)
_u_ext[-1] = 0.0  # right wall (fixed)


def acceleration(u: np.ndarray) -> np.ndarray:
    """
    Acceleration of each mass from its two nearest-neighbor springs.

    Walls are fixed at displacement zero. When ALPHA is zero this is pure
    Hooke's law; nonzero ALPHA adds the FPU quadratic correction term.
    """
    _u_ext[1:-1] = u
    du_right = _u_ext[2:] - _u_ext[1:-1]  # stretch of the right spring
    du_left = _u_ext[1:-1] - _u_ext[:-2]  # stretch of the left spring
    f_right = K * du_right + ALPHA * K * du_right**2
    f_left = K * du_left + ALPHA * K * du_left**2
    return (f_right - f_left) / M


def integrate(u0, v0, tf, ts):
    """Yoshida 4th-order symplectic integration; returns (t_hist, u_hist)."""
    dt = tf / ts
    cs, ds = yoshida_coeffs()

    t_hist = np.zeros(ts)
    u_hist = np.zeros((ts, N))
    u = u0.copy()
    v = v0.copy()
    u_hist[0] = u

    for step in range(1, ts):
        u = u + cs[0] * v * dt
        for j in range(3):
            v = v + ds[j] * acceleration(u) * dt
            u = u + cs[j + 1] * v * dt
        t_hist[step] = step * dt
        u_hist[step] = u
    return t_hist, u_hist


# Quick check: pluck mass 1 forward and confirm it is pulled back toward zero
u_test = np.zeros(N)
u_test[0] = 0.3
a_test = acceleration(u_test)
print(f"displacement u = {u_test}")
print(f"acceleration a = {a_test}")
print(f"mass 1 accel   = {a_test[0]:+.2f} m/s^2  (negative: restoring force)")

## Integrating the chain and reading its normal modes

We run two integrations from the same plucked initial state:

- a **short run** (0-10 s) for the time-domain plots, and
- a **long run** (0-100 s) whose length sets the FFT frequency resolution to
  $1/100 = 0.01\ \text{Hz}$.

A linear chain has exactly $N$ special patterns of motion called **normal
modes**. In a normal mode every mass oscillates at the *same* single frequency,
and for a chain between fixed walls those frequencies are

$$\omega_j = 2\sqrt{\tfrac{k}{m}}\;
             \sin\!\left(\frac{j\pi}{2(N+1)}\right),
             \qquad j = 1, 2, \dots, N.$$

Because the chain is linear, energy placed in one mode **stays** in that mode
forever - the modes never exchange energy. When we pluck a single mass we
actually excite a mixture of all five modes, so the motion looks complicated. The
FFT untangles it: taking the power spectrum of each mass's motion reveals sharp
peaks at exactly these five frequencies.

The three-panel figure below shows:

1. **Top:** displacement of each mass from equilibrium vs time.
2. **Middle:** absolute position vs time, with the two fixed walls drawn in
   distinct line styles - the left wall **dashed** and the right wall
   **dash-dot** so they cannot be confused - and mass 1's zero-crossings marked.
3. **Bottom:** a grouped bar chart of the FFT power each mass carries at each of
   the five normal-mode frequencies - the fingerprint of the linear chain.

### Why some masses are missing at certain frequencies

Look ahead at the bottom bar chart and you will notice that **not every mass has
a bar at every frequency**. This is not a bug or a rounding artifact - it is the
geometry of the normal modes.

Each normal mode $j$ has a fixed spatial shape. The amplitude of mode $j$ at
mass $i$ is

$$\text{shape}_j(i) = \sin\!\left(\frac{j\,\pi\,i}{N+1}\right),
  \qquad i = 1, 2, \dots, N.$$

Wherever that sine is **zero**, mass $i$ sits on a **node** of mode $j$: it does
not move at all when the chain vibrates in that mode, so it carries **no power**
at that mode's frequency and its bar vanishes (it falls below the -50 dB floor of
the plot). For our $N = 5$ chain the nodes work out as:

| Mode $j$ | Frequency | Masses on a node (silent) |
|---|---|---|
| 1 | lowest  | none - every mass moves |
| 2 |         | mass 3 |
| 3 |         | masses 2 and 4 |
| 4 |         | mass 3 |
| 5 | highest | none - every mass moves |

For example, in mode 3 the shape is $\sin(\pi i/2)$, which is exactly zero at
$i = 2$ and $i = 4$, so masses 2 and 4 are stationary and contribute nothing at
the mode-3 frequency. The slowest and fastest modes (1 and 5) have no interior
nodes, so all five masses show up there.

So a missing bar is a positive result: it is direct evidence of the standing-wave
node pattern that defines each normal mode.

In [ ]:
# Cell 03 - Integrate the linear chain and plot its normal-mode spectrum

# Initial conditions: pluck the first mass forward, everything else at rest
u0 = np.zeros(N)
v0 = np.zeros(N)
u0[0] = 0.3

# Short run for the time-domain panels (0-10 s)
tf_plot, ts_plot = 10.0, 20_000
print("Integrating short run for time-domain plots...")
t_hist, u_hist = integrate(u0, v0, tf_plot, ts_plot)

# Long run for the FFT (0-100 s gives 0.01 Hz frequency resolution)
tf_fft, ts_fft = 100.0, 50_000
print("Integrating long run for the FFT...")
_, u_hist_fft = integrate(u0, v0, tf_fft, ts_fft)
dt_fft = tf_fft / ts_fft
print("done")

# Predicted normal-mode frequencies (Hz) for comparison
modes = np.arange(1, N + 1)
omega_modes = 2.0 * np.sqrt(K / M) * np.sin(modes * np.pi / (2 * (N + 1)))
f_modes = omega_modes / (2 * np.pi)
print("Predicted normal-mode frequencies (Hz):", np.round(f_modes, 3))

_, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 9), num="fpu_spring_mass")
ax1.sharex(ax2)
colors = ["crimson", "royalblue", "darkorange", "forestgreen", "mediumpurple"]
ax1.tick_params(axis="x", labelbottom=True)
ax1.set_xlabel("Time (sec)")
ax1.xaxis.set_major_locator(plt.MultipleLocator(1.0))

# Top panel: displacement from equilibrium vs time
for i in range(N):
    ax1.plot(t_hist, u_hist[:, i], lw=1.5, color=colors[i], label=f"mass {i + 1}")
ax1.set_ylabel("Displacement from equilibrium (m)")
ax1.set_title(
    rf"Linear Spring-Mass Chain - 4th-Order Symplectic (Yoshida)  $\alpha={ALPHA}$"
)
ax1.legend(loc="upper right", fontsize=8, framealpha=1.0, facecolor="white")
ax1.axhline(0, color="gray", lw=0.5)
ax1.grid(True)

# Middle panel: absolute position vs time with the fixed walls.
# The two walls use different dash patterns so they can be told apart:
# the left wall is dashed and the right wall is dash-dot.
for i in range(N):
    eq_pos = (i + 1) * A
    ax2.plot(
        t_hist,
        eq_pos + u_hist[:, i],
        lw=1.5,
        color=colors[i],
        label=f"mass {i + 1} (eq = {eq_pos:.0f} m)",
    )
ax2.axhline(0, color="black", lw=2, ls="--", label="left wall (dashed)")
ax2.axhline((N + 1) * A, color="black", lw=2, ls="-.", label="right wall (dash-dot)")
u1 = u_hist[:, 0]
zero_crossings = np.where(np.diff(np.sign(u1)))[0]
ax2.vlines(t_hist[zero_crossings], ymin=0.5, ymax=5.5, color="black", ls=":")
ax2.set_xlabel("Time (sec)")
ax2.xaxis.set_major_locator(plt.MultipleLocator(1.0))
ax2.set_ylabel("Absolute x position (m)")
ax2.legend(loc="upper right", fontsize=8, framealpha=1.0, facecolor="white")
ax2.grid(True)

# Bottom panel: grouped bar chart of peak power at each normal-mode frequency
freqs = fftfreq(ts_fft, dt_fft)
pos_mask = freqs > 0
window = np.hanning(ts_fft)
window_gain = window.sum()
ref_freqs = freqs[pos_mask]

all_spectra_db = []
for i in range(N):
    signal = u_hist_fft[:, i]
    spectrum = np.abs(fft(signal * window)) * 2 / window_gain
    power_db = 20 * np.log10(np.maximum(spectrum, 1e-15))
    all_spectra_db.append(power_db[pos_mask])

# Find the N tallest, well-separated peaks from the measured spectra
all_spectra_array = np.array(all_spectra_db)
ref_spectrum = all_spectra_array.max(axis=0)
freq_resolution = ref_freqs[1] - ref_freqs[0]
min_separation = int(0.2 / freq_resolution)
peak_indices, _ = find_peaks(ref_spectrum, height=-40, distance=min_separation)
peak_indices = peak_indices[np.argsort(ref_spectrum[peak_indices])[::-1]][:N]
peak_indices = np.sort(peak_indices)
peak_freqs = ref_freqs[peak_indices]

bar_width = 0.012
offsets = np.linspace(-(N - 1) / 2, (N - 1) / 2, N) * bar_width * 1.1
bottom_db = -50
for i in range(N):
    peak_powers = np.array([all_spectra_db[i][idx] for idx in peak_indices])
    bar_heights = peak_powers - bottom_db
    ax3.bar(
        peak_freqs + offsets[i],
        bar_heights,
        width=bar_width,
        color=colors[i],
        label=f"mass {i + 1}",
        bottom=bottom_db,
        align="center",
    )
ax3.set_xlabel("Frequency (Hz)")
ax3.set_ylabel("Power (dB)")
ax3.set_ylim(-50, -10)
ax3.set_xticks(peak_freqs)
ax3.set_xticklabels([f"{f:.3f} Hz" for f in peak_freqs], fontsize=8)
ax3.set_title("Normal Mode Power by Mass", fontsize=10)
ax3.legend(loc="upper right", fontsize=8, framealpha=1.0, facecolor="white")
ax3.grid(True, axis="y")

plt.tight_layout()
plt.show()

print("Measured FFT peak frequencies (Hz):", np.round(peak_freqs, 3))

---
# Part 2 - The Nonlinear FPU-$\alpha$ Recurrence

Now we reproduce the historic 1953 experiment. Two things change from Part 1, so
this section **redefines the chain parameters and the acceleration function**
before using them:

| Parameter | Part 1 (linear) | Part 2 (FPU-$\alpha$) |
|---|---|---|
| Number of masses $N$ | 5 | **32** |
| Spring constant $k$ | 50 | **1** |
| Nonlinearity $\alpha$ | 0 | **0.7** |

The 32-mass chain gives a cleaner recurrence, and $\alpha = 0.7$ turns on the
quadratic correction to the spring force:

$$F = k\,(\Delta u) + \alpha\,k\,(\Delta u)^2,
    \qquad \Delta u = u_{i+1} - u_i.$$

That extra $(\Delta u)^2$ term is the whole story. In a purely linear chain the
normal modes are exactly independent - energy dropped into mode 1 stays in mode 1
forever, and nothing interesting happens. The nonlinearity **couples** the modes,
letting them trade energy. Fermi expected this coupling to smear the energy
evenly across all 32 modes (thermalization). The redefinition below sets up the
tools to test that expectation.

Because these globals share the notebook namespace with Part 1, we rebuild
`_u_ext`, the normal-mode projection matrix, and `acceleration` from scratch here
so nothing stale from the 5-mass chain leaks in.

In [ ]:
# Cell 04 - Part 2 redefines the chain parameters and the mode projection

# Part 2 physical parameters (these REPLACE the Part 1 globals)
N = 32  # number of masses (more masses = cleaner recurrence)
K = 1.0  # spring constant (N/m)
M = 1.0  # mass of each particle (kg)
ALPHA = 0.7  # FPU nonlinearity strength
A = 1.0  # equilibrium spacing (m)

# Rebuild the padded position array for the new N
_u_ext = np.empty(N + 2)
_u_ext[0] = 0.0  # left wall (fixed)
_u_ext[-1] = 0.0  # right wall (fixed)


def acceleration(u: np.ndarray) -> np.ndarray:
    """FPU-alpha acceleration: Hooke's law plus a quadratic correction term."""
    _u_ext[1:-1] = u
    du_right = _u_ext[2:] - _u_ext[1:-1]
    du_left = _u_ext[1:-1] - _u_ext[:-2]
    f_right = K * du_right + ALPHA * K * du_right**2
    f_left = K * du_left + ALPHA * K * du_left**2
    return (f_right - f_left) / M


# Normal-mode projection matrix and angular frequencies for the linear chain.
# mode_matrix[j, i] = sqrt(2/(N+1)) * sin((j+1) * pi * (i+1) / (N+1))
_j_idx = np.arange(1, N + 1).reshape(N, 1)
_i_idx = np.arange(1, N + 1).reshape(1, N)
mode_matrix = np.sqrt(2.0 / (N + 1)) * np.sin(_j_idx * np.pi * _i_idx / (N + 1))
omega = 2.0 * np.sqrt(K / M) * np.sin(_j_idx.ravel() * np.pi / (2 * (N + 1)))
omega_sq = omega**2


def mode_energies(u: np.ndarray, v: np.ndarray) -> np.ndarray:
    """Energy in each linear normal mode: E_j = (P_j^2 + omega_j^2 Q_j^2) / 2."""
    Q = mode_matrix @ u  # normal-mode coordinates
    P = mode_matrix @ v  # normal-mode momenta
    return 0.5 * (P**2 + omega_sq * Q**2)


def total_energy(u: np.ndarray, v: np.ndarray) -> float:
    """
    True conserved energy of the nonlinear chain: kinetic plus the FULL spring
    potential, including the anharmonic (cubic) term that the modal energies omit.
    Each spring's potential is V = (1/2) k du^2 + (alpha k / 3) du^3.
    """
    _u_ext[1:-1] = u
    du = _u_ext[1:] - _u_ext[:-1]  # stretch of every spring, walls included
    kinetic = 0.5 * M * np.sum(v**2)
    potential = np.sum(0.5 * K * du**2 + (ALPHA * K / 3.0) * du**3)
    return kinetic + potential


# Quick check: put the chain in the shape of mode 1 and confirm the energy
# lands almost entirely in mode 1
indices = np.arange(1, N + 1)
u_mode1 = np.sin(1 * np.pi * indices / (N + 1))
E_check = mode_energies(u_mode1, np.zeros(N))
print(f"N = {N}, ALPHA = {ALPHA}, K = {K}")
print(f"Energy in mode 1 : {E_check[0]:.4f} J")
print(f"Energy in modes 2+: {E_check[1:].sum():.2e} J  (should be ~0)")
print(f"True total energy : {total_energy(u_mode1, np.zeros(N)):.4f} J")

## Watching the energy leave - and come back

We now excite **only normal mode 1** (the slowest, longest-wavelength vibration)
by shaping the initial displacement as $u_i = \sin(\pi i / (N+1))$ with every mass
at rest. All the energy starts in a single mode. Then we integrate the nonlinear
chain and, at regular intervals, project the state onto the normal modes to see
where the energy has gone.

**A note on the run length.** The original script took one million time steps,
which is far too slow to run inside a notebook. We keep the same final time
$t_f = 2\pi\cdot 1000 \approx 6283\ \text{s}$ (long enough for the recurrence to
play out) but use fewer steps, so the step size $\Delta t$ grows. Because the
Yoshida method is 4th-order and symplectic, it stays both stable and
energy-conserving at the larger step - which the flat total-energy panel confirms.

The two panels show:

1. **Top:** energy in the first six normal modes vs time. Watch energy drain out
   of mode 1 into modes 2, 3, ... and then, instead of spreading evenly, come
   **rushing back** into mode 1. That return is the **FPU recurrence** - the chain
   does *not* thermalize.
2. **Bottom:** the **true total energy** of the chain - kinetic energy plus the
   full nonlinear spring potential. With a good symplectic integrator this line
   must stay essentially **flat**; any visible slope would mean the integrator,
   not the physics, was creating or destroying energy.

One subtlety worth naming: the *modal* energies in the top panel are the energies
of the underlying **linear** modes, and their sum is **not** quite the full energy
- it leaves out the anharmonic ($\alpha$) part of the spring potential. Some
energy is briefly parked in that nonlinear term whenever the springs are
stretched, so the modal sum alone would wiggle by a few percent even with a
perfect integrator. The bottom panel therefore plots the complete conserved
Hamiltonian, which is the honest test of the integrator.

In [ ]:
# Cell 05 - Integrate the FPU-alpha chain and watch the recurrence

# Simulation parameters
tf = 2 * np.pi * 1000  # final time (s) - long enough to see the recurrence
ts = 200_000  # time steps (reduced from 1e6 so the notebook runs quickly)
dt = tf / ts
print(f"tf = {tf:.1f} s, ts = {ts}, dt = {dt:.5f} s")

# Initial conditions: excite ONLY normal mode 1, the classic FPU setup
mode = 1
amplitude = 1.0
indices = np.arange(1, N + 1)
u = amplitude * np.sin(mode * np.pi * indices / (N + 1))
v = np.zeros(N)

# Yoshida 4th-order coefficients
cs, ds = yoshida_coeffs()

# Sample the modal energies every sample_every steps (matmul is cheap, but we
# do not need one sample per step to draw a smooth curve)
sample_every = 200
n_samples = ts // sample_every
t_hist = np.zeros(n_samples)
energy_hist = np.zeros((n_samples, N))  # per-mode (linear) energies
total_hist = np.zeros(n_samples)  # true conserved Hamiltonian
sample_idx = 0

print("Integrating...")
for step in range(ts):
    if step % sample_every == 0:
        t_hist[sample_idx] = step * dt
        energy_hist[sample_idx] = mode_energies(u, v)
        total_hist[sample_idx] = total_energy(u, v)
        sample_idx += 1
    u = u + cs[0] * v * dt
    for j in range(3):
        v = v + ds[j] * acceleration(u) * dt
        u = u + cs[j + 1] * v * dt
print("done")

# Report energy conservation (the key symplectic sanity check)
e_mean = total_hist.mean()
drift = np.ptp(total_hist)  # peak-to-peak spread of the TRUE energy
print(f"Mean total energy    : {e_mean:.6f} J")
print(f"Total-energy spread  : {drift:.2e} J  ({100 * drift / e_mean:.6f}% of mean)")

_, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), num="fpu_recurrence")

# Top panel: energy in the first several modes vs time
n_modes_plot = 6
colors = [
    "crimson",
    "royalblue",
    "darkorange",
    "forestgreen",
    "mediumpurple",
    "saddlebrown",
]
for j in range(n_modes_plot):
    ax1.plot(t_hist, energy_hist[:, j], lw=1.5, color=colors[j], label=f"mode {j + 1}")
ax1.set_ylabel("Energy in mode (J)")
ax1.set_title(
    rf"FPU Recurrence ({N} masses, $\alpha$ = {ALPHA}), Yoshida 4th-order symplectic"
)
ax1.legend(loc="center right", fontsize=8, framealpha=1.0, facecolor="white")
ax1.grid(True)

# Bottom panel: true total energy - should be flat (symplectic conservation check)
ax2.plot(t_hist, total_hist, lw=1.0, color="black")
ax2.set_xlabel("Time (s)")
ax2.set_ylabel("Total energy (J)")
ax2.set_title(
    "True total energy - kinetic + full nonlinear potential (conservation check)"
)
ax2.grid(True)

# Zoom the y-axis tight around the mean to expose any drift
e_range = max(np.ptp(total_hist) * 5, e_mean * 1e-6)
ax2.set_ylim(e_mean - e_range, e_mean + e_range)

plt.tight_layout()
plt.show()

## What just happened

The top panel tells the story Fermi did not expect. Energy poured out of mode 1
into a small group of low modes, the chain looked briefly as though it might be
heading toward an even spread - and then the flow reversed and mode 1 recovered
almost all of its original energy. The system is nonlinear, yet it behaves almost
like a set of independent oscillators that periodically re-align.

The bottom panel is the control experiment. The total energy stays flat to a
tiny fraction of a percent, which means the energy sloshing between modes in the
top panel is **real physics**, not numerical drift from the integrator. This is
exactly why a symplectic method matters here: an ordinary Runge-Kutta solver
would slowly leak energy over these millions of substeps and could easily fake -
or destroy - the recurrence.

The FPU recurrence was the first hint that nonlinear systems can hide beautiful
order where thermodynamic intuition predicts chaos. Explaining it led directly to
the discovery of **solitons** and to the modern theory of integrable systems -
one of the most productive surprises in the history of computational physics.